## Notebook to train the shared cell LSTM

In [7]:
from pathlib import Path
import sys

import numpy as np

# Run this notebook with LSTM/ as the working directory.
PROJECT_ROOT = Path("..")
sys.path.insert(0, "../data/simulated")

from p0 import p0_MC as estimate_p0

# Monte Carlo call price parameters (same rough-vol model as the dataset).
S0 = 100.0
T = 30 / 365
N = 30
H = 0.10
V0 = 0.235**2
nu = 1.9
rho = -0.7
strike = 100.0
MC_PATHS = 1_000_000

p0 = estimate_p0(
    M=MC_PATHS,
    K=strike,
    batch_size=2_000,
    S0=S0, T=T, N=N, H=H, V0=V0, nu=nu, rho=rho,
)
print(f"Monte Carlo call price p0 (M={MC_PATHS:,}): {p0:.6f}")

Monte Carlo call price p0 (M=1,000,000): 2.423414


In [8]:
import torch
from torch.utils.data import DataLoader, TensorDataset

# The saved paths have shape [number of paths, N+1, 2], with
# columns [stock price, forward variance].
train_array = np.load("../data/simulated/alpha_-0.4/N_30/train.npy")
test_array = np.load("../data/simulated/alpha_-0.4/N_30/test.npy")

# Keep validation paths separate from the held-out test set.
split_at = int(0.9 * len(train_array))
train_paths = torch.from_numpy(train_array[:split_at]).float()
val_paths = torch.from_numpy(train_array[split_at:]).float()
test_paths = torch.from_numpy(test_array).float()

BATCH_SIZE = 2_048
train_loader = DataLoader(
    TensorDataset(train_paths), batch_size=BATCH_SIZE, shuffle=True
)
print(f"Loaded {len(train_paths):,} training, {len(val_paths):,} validation, and {len(test_paths):,} test paths.")
print(f"Each path has shape {tuple(train_paths.shape[1:])}.")

Loaded 90,000 training, 10,000 validation, and 20,000 test paths.
Each path has shape (31, 2).


In [9]:
import importlib
import shared_cell

# Pick up edits to the model even in an already running notebook kernel.
importlib.reload(shared_cell)
from shared_cell import LSTMHedger, hedging_loss

# Train the shared-cell hedger and save a reloadable checkpoint.
HIDDEN_SIZE  = 32
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = LSTMHedger(hidden_size=HIDDEN_SIZE, strike=strike, log_moneyness=True).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

option_price = float(p0)
EPOCHS = 75
for epoch in range(EPOCHS):
    model.train()
    train_loss_sum = 0.0
    train_count = 0

    for (prices,) in train_loader:
        prices = prices.to(device)
        optimizer.zero_grad()
        loss = hedging_loss(model, prices, strike, option_price)
        loss.backward()
        optimizer.step()
        train_loss_sum += loss.item() * len(prices)
        train_count += len(prices)

    model.eval()
    with torch.no_grad():
        validation_loss = hedging_loss(
            model, val_paths.to(device), strike, option_price
        )

    print(
        f"Epoch {epoch + 1:03d}/{EPOCHS}: "
        f"train={train_loss_sum / train_count:.6f}, "
        f"validation={validation_loss.item():.6f}"
    )

CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
checkpoint_path = CHECKPOINT_DIR / f"shared_cell_lstm_alpha_{H - 0.5:.1f}_N_{N}.pt"
torch.save(
    {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "epoch": EPOCHS,
        "hidden_size": HIDDEN_SIZE,
        "strike": strike,
        "option_price": option_price,
        "alpha": H - 0.5,
        "N": N,
    },
    checkpoint_path,
)
print(f"Saved model checkpoint to {checkpoint_path}")

Epoch 001/75: train=4.520545, validation=3.994655
Epoch 002/75: train=3.869648, validation=3.624450
Epoch 003/75: train=2.972126, validation=2.452713
Epoch 004/75: train=2.360214, validation=2.238592
Epoch 005/75: train=2.157083, validation=2.096017
Epoch 006/75: train=2.016110, validation=1.987354
Epoch 007/75: train=1.909452, validation=1.878642
Epoch 008/75: train=1.817657, validation=1.808576
Epoch 009/75: train=1.751445, validation=1.744401
Epoch 010/75: train=1.686527, validation=1.666698
Epoch 011/75: train=1.631296, validation=1.614275
Epoch 012/75: train=1.586126, validation=1.559299
Epoch 013/75: train=1.539151, validation=1.541121
Epoch 014/75: train=1.481691, validation=1.465788
Epoch 015/75: train=1.439478, validation=1.430193
Epoch 016/75: train=1.397546, validation=1.404280
Epoch 017/75: train=1.371285, validation=1.348382
Epoch 018/75: train=1.340484, validation=1.324038
Epoch 019/75: train=1.294067, validation=1.280825
Epoch 020/75: train=1.261230, validation=1.247350
